In [1]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.cluster import AgglomerativeClustering
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

d:\git\Taxonomy_Buidling_Textual_Corpora\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_json("D:/git/Taxonomy_Buidling_Textual_Corpora/data/icecat_data_train.json")

In [3]:
df_small = df.head(1000).copy()
df_small.shape

(1000, 45)

In [4]:
df_small

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392099,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37079,[],1056,EN,Flat Panel Wall Mounts,...,None,None,NaN,None,None,None,None,NaN,2833>220>1056,Computers & Electronics>TVs & Monitors>Flat Pa...
1008636,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,SIC1297184LCD0,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
581225,Middle Atlantic Products,,https://images.icecat.biz/img/brand/thumb/2702...,Middle Atlantic Products,https://images.icecat.biz/img/brand/thumb/2702...,WUSS14,None,1532,EN,Rack Accessories,...,None,None,1967411.0,EN,2019-04-17 12:12:17,[Holds components too wide for standard racksh...,None,NaN,2833>106>236>1532,Computers & Electronics>Computer Components>Ch...
301585,Philips,,https://images.icecat.biz/img/brand/thumb/25_b...,Philips,https://images.icecat.biz/img/brand/thumb/25_b...,55PUS6272/12,None,1584,EN,TVs,...,None,None,417921.0,EN,2018-02-01 09:50:04,"[139 cm (55""), 4K Ultra-HD LED TV, Quad Core, ...",None,NaN,2833>220>1584,Computers & Electronics>TVs & Monitors>TVs


STEP 1 — Extract A, B, C Taxonomy Levels

Use a robust splitter that handles:
2-level
3-level
4+ level paths
(some Icecat paths have 4 levels!)

In [54]:
def split_taxonomy(path):
    """
    Handle 2, 3, 4+ levels in pathlist_names.
    A = root
    B = second level
    C = everything below B (joined back with '>').
    """
    if pd.isna(path):
        return pd.Series([None, None, None])

    parts = [p.strip() for p in str(path).split(">")]

    # A > B
    if len(parts) == 2:
        return pd.Series([parts[0], parts[1], None])

    # A > B > C
    if len(parts) == 3:
        return pd.Series([parts[0], parts[1], parts[2]])

    # A > B > C > D --> treat C as "C > D"
    return pd.Series([parts[0], parts[1], ">".join(parts[2:])])


In [55]:
df_small[["A", "B", "C"]] = df_small["pathlist_names"].apply(split_taxonomy)
df_small[["pathlist_names", "A", "B", "C"]].head(10)

,pathlist_names,A,B,C
1072689,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations
906402,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories>Notebook Spare Parts
411281,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,Computer Cables,Fibre Optic Cables
425903,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts
1047582,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations
904910,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories>Notebook Spare Parts
157385,Computers & Electronics>Software>Software Lice...,Computers & Electronics,Software,Software Licenses/Upgrades
934548,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories>Notebook Spare Parts
876762,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories>Notebook Spare Parts
397028,Computers & Electronics>Telecom & Navigation>M...,Computers & Electronics,Telecom & Navigation,Mobile Phone Spare Parts


In [56]:
# df_small = first 1000 rows (for now)
# and you already ran:
df_small[["A", "B", "C"]] = df_small["pathlist_names"].apply(split_taxonomy)


In [57]:
#Global counts: how many A, B, C?
print("Total A-level categories:", df_small["A"].nunique())
print("Total B-level categories:", df_small["B"].nunique())
print("Total C-level categories:", df_small["C"].nunique())


Total A-level categories: 1
Total B-level categories: 16
Total C-level categories: 143


In [58]:
#For each A: how many B and C?
# B per A
A_B_counts = (
    df_small.groupby("A")["B"]
    .nunique()
    .reset_index(name="num_B_under_A")
)

# C per A
A_C_counts = (
    df_small.groupby("A")["C"]
    .nunique()
    .reset_index(name="num_C_under_A")
)

A_summary = A_B_counts.merge(A_C_counts, on="A")
A_summary


,A,num_B_under_A,num_C_under_A
0,Computers & Electronics,16,143


In [59]:
print("=== A-level summary (how many B and C under each A) ===")
for _, row in A_summary.iterrows():
    print(f"A: {row['A']}")
    print(f"  B-level subcategories: {row['num_B_under_A']}")
    print(f"  C-level subcategories: {row['num_C_under_A']}")
    print()


=== A-level summary (how many B and C under each A) ===
A: Computers & Electronics
  B-level subcategories: 16
  C-level subcategories: 143



In [50]:
# For each B: how many C and how many products?
# C per B (within each A)
B_C_counts = (
    df_small.groupby(["A", "B"])["C"]
    .nunique()
    .reset_index(name="num_C_under_B")
)

# products per B
B_prod_counts = (
    df_small.groupby(["A", "B"])["ProductName"]
    .count()
    .reset_index(name="num_products_under_B")
)

B_summary = B_C_counts.merge(B_prod_counts, on=["A", "B"])
B_summary


,A,B,num_C_under_B,num_products_under_B
0,Computers & Electronics,Batteries & Power Supplies,7,27
1,Computers & Electronics,Computer Cables,11,42
2,Computers & Electronics,Computer Components,14,65
3,Computers & Electronics,Computers,15,450
4,Computers & Electronics,Consumer Audio & Video Equipment,13,25
5,Computers & Electronics,Data Input Devices,3,25
6,Computers & Electronics,Data Storage,11,59
7,Computers & Electronics,Networking,7,11
8,Computers & Electronics,Office Electronics,1,1
9,Computers & Electronics,Photo & Video Equipment,8,14


In [61]:
print("=== B-level summary (per A) ===")
for (a, b), sub in B_summary.groupby(["A", "B"]):
    row = sub.iloc[0]
    print(f"A: {a}")
    print(f"  B: {b}")
    print(f"    #C-level subcategories: {row['num_C_under_B']}")
    print(f"    #products (total under this B): {row['num_products_under_B']}")
    print()


=== B-level summary (per A) ===
A: Computers & Electronics
  B: Batteries & Power Supplies
    #C-level subcategories: 7
    #products (total under this B): 27

A: Computers & Electronics
  B: Computer Cables
    #C-level subcategories: 11
    #products (total under this B): 42

A: Computers & Electronics
  B: Computer Components
    #C-level subcategories: 14
    #products (total under this B): 65

A: Computers & Electronics
  B: Computers
    #C-level subcategories: 15
    #products (total under this B): 450

A: Computers & Electronics
  B: Consumer Audio & Video Equipment
    #C-level subcategories: 13
    #products (total under this B): 25

A: Computers & Electronics
  B: Data Input Devices
    #C-level subcategories: 3
    #products (total under this B): 25

A: Computers & Electronics
  B: Data Storage
    #C-level subcategories: 11
    #products (total under this B): 59

A: Computers & Electronics
  B: Networking
    #C-level subcategories: 7
    #products (total under this B): 1

In [64]:
#For each C: how many products?
C_prod_counts = (
    df_small.groupby(["A", "B", "C"])["ProductName"]
    .count()
    .reset_index(name="num_products_under_C")
)

C_prod_counts  # inspect first 20


,A,B,C,num_products_under_C
0,Computers & Electronics,Batteries & Power Supplies,Household Batteries,2
1,Computers & Electronics,Batteries & Power Supplies,Power Adapters & Inverters,14
2,Computers & Electronics,Batteries & Power Supplies,Power Banks,1
3,Computers & Electronics,Batteries & Power Supplies,Power Distribution Units (PDUs),1
4,Computers & Electronics,Batteries & Power Supplies,Power Supply Units,3
...,...,...,...,...
138,Computers & Electronics,Telecom & Navigation,Screen Protectors,8
139,Computers & Electronics,Telecom & Navigation,Smartphones,6
140,Computers & Electronics,Telecom & Navigation,Telephony Equipment>Telephones,2
141,Computers & Electronics,Warranty & Support,IT Support Services,2


In [62]:
print("=== C-level summary (A → B → C → #products) ===")
for (a, b), sub in C_prod_counts.groupby(["A", "B"]):
    print(f"\nA: {a}")
    print(f"  B: {b}")
    for _, row in sub.iterrows():
        print(f"    C: {row['C']}  ({row['num_products_under_C']} products)")


=== C-level summary (A → B → C → #products) ===

A: Computers & Electronics
  B: Batteries & Power Supplies
    C: Household Batteries  (2 products)
    C: Power Adapters & Inverters  (14 products)
    C: Power Banks  (1 products)
    C: Power Distribution Units (PDUs)  (1 products)
    C: Power Supply Units  (3 products)
    C: UPS Batteries  (1 products)
    C: Uninterruptible Power Supplies (UPSs)  (5 products)

A: Computers & Electronics
  B: Computer Cables
    C: Cable Interface/Gender Adapters  (4 products)
    C: Cable Protectors  (1 products)
    C: DisplayPort Cables  (1 products)
    C: Fibre Optic Cables  (7 products)
    C: HDMI Cables  (2 products)
    C: Networking Cables  (18 products)
    C: Power Cables  (3 products)
    C: SCSI Cables  (1 products)
    C: USB Cables  (3 products)
    C: Video Cable Adapters  (1 products)
    C: Wire Connectors  (1 products)

A: Computers & Electronics
  B: Computer Components
    C: Chassis Components>Computer Case Parts  (1 products

In [63]:
def print_full_hierarchy(df):
    # Loop over A-level categories
    for a in df["A"].dropna().unique():
        print(f"\nA: {a}")
        print("=" * 80)

        df_a = df[df["A"] == a]

        # Loop over B-level categories within this A
        for b in df_a["B"].dropna().unique():
            print(f"  B: {b}")

            df_b = df_a[df_a["B"] == b]

            # Loop over C-level categories within this B
            for c in df_b["C"].dropna().unique():
                df_c = df_b[df_b["C"] == c]

                print(f"    C: {c}  ({len(df_c)} products)")

                # Print product names under this C
                for pname in df_c["ProductName"].dropna().tolist():
                    print(f"        • {pname}")

            print()  # space after each B section

# Run it on the sample
print_full_hierarchy(df_small)



A: Computers & Electronics
  B: Computers
    C: PCs/Workstations  (38 products)
        • K31CD-IT049T
        • C30
        • UN42-M031M
        • P310
        • 875-1303ng
        • 400 G6
        • M910q
        • 3 VR7RD-037US
        • 400 G4 + EliteDisplay E223
        • ProDesk 600 G3 Desktop Mini PC
        • G11CD-FR139T
        • S2660G
        • h8-1500et
        • Compaq Presario SG3320IL Desktop PC
        • TC-710
        • XC605
        • DM500T4Z
        • M32CD-IT051T
        • A 8RC-290FR
        • Z640
        • Compaq 6200 Pro Microtower PC (ENERGY STAR)
        • Z440
        • P500
        • 550-131ns
        • 5680
        • P420 E85+
        • M83
        • 595-p0059nl
        • K20CD-KR002D
        • 690-0043ns
        • Aegis Ti3 VR7RD
        • S510 + 19.5" LED ThinkVision E2054
        • 800 G4 + LaserJet Pro M15w
        • 800 G4
        • M73
        • Y700-34ISH
        • M770
        • 690-0999nf
    C: Notebook Parts & Accessories>Notebook Spare Parts